# `jupyter_builder` smoke test

In [ ]:
from jupyter_builder.main import (
    BuilderApp,
    BuildLabExtensionApp,
    DevelopLabExtensionApp,
    WatchLabExtensionApp,
)

assert sorted(BuilderApp.subcommands) == ["build", "develop", "watch"]

for cls in (BuildLabExtensionApp, DevelopLabExtensionApp, WatchLabExtensionApp):
    instance = cls()
    assert callable(instance.initialize) and callable(instance.start)
    print(f"{cls.__name__}: {instance.description.splitlines()[0]}")
print("OK — BuilderApp exposes build/develop/watch subcommand apps")

In [ ]:
import json, os, pathlib, tempfile
from jupyter_builder.federated_extensions import develop_labextension

with tempfile.TemporaryDirectory() as td:
    root = pathlib.Path(td)
    ext = root / "my-extension"
    ext.mkdir()
    (ext / "package.json").write_text(
        json.dumps({"name": "my-extension", "version": "1.2.3", "jupyterlab": {"extension": True}})
    )
    labdir = root / "labextensions"
    labdir.mkdir()
    dest = develop_labextension(str(ext), labextensions_dir=str(labdir))
    assert os.path.islink(dest), dest
    assert os.path.realpath(dest) == str(ext.resolve())
    print("symlinked:", dest)
print("OK — develop-mode install created the expected symlink")

In [ ]:
from jupyter_builder import jupyterlab_semver as semver

v = semver.make_semver("1.2.3-rc.1", False)
assert (v.major, v.minor, v.patch, v.prerelease) == (1, 2, 3, ["rc", 1])
assert str(semver.valid("1.2.3", False)) == "1.2.3"  # valid() returns a SemVer object
assert semver.parse("not-a-version", False) is None  # parse (not valid) is the safe reject path
assert semver.inc("1.2.3", "minor", False) == "1.3.0"
assert semver.satisfies("1.4.0", "^1.2.0", False)
assert not semver.satisfies("2.0.0", "^1.2.0", False)
assert semver.gt("2.0.0", "1.9.9", False) and semver.lt("1.0.0-alpha", "1.0.0", False)
rng = semver.Range(">=1.2.0 <2.0.0", False)
assert rng.test(semver.make_semver("1.5.0", False))
assert semver.max_satisfying(["1.1.0", "1.9.0", "2.1.0"], "^1.0.0", False) == "1.9.0"
print("OK — jupyterlab_semver parse/inc/satisfies/Range all behave")

In [ ]:
import pathlib, subprocess
from jupyter_builder.jlpm import YARN_PATH

yarn = pathlib.Path(YARN_PATH)
assert yarn.is_file(), f"bundled yarn.js missing: {yarn}"
assert yarn.stat().st_size > 10_000, "bundled yarn.js suspiciously small"
print(f"bundled yarn.js: {yarn} ({yarn.stat().st_size} bytes)")

proc = subprocess.run(
    ["jupyter-builder", "--help"], capture_output=True, text=True, timeout=120
)
assert proc.returncode == 0, proc.stderr
for sub in ("build", "develop", "watch"):
    assert sub in proc.stdout
print("OK — `jupyter-builder --help` lists build/develop/watch")